# Induction heads and ablations

Score every attention head in `attn-only-2l` for induction and previous-token attention, then zero-ablate the top induction heads and a random control set to see how much of the copying benefit each removes. This experiment builds on `01_observe_copying.ipynb`; it does not repeat the baseline behavioral measurement.

## Metric

For each scored second-block query position, the **induction score** is the mean attention paid to the key holding the token the query should copy (the token after the earlier occurrence of the current token). The **previous-token score** is the mean attention paid to the immediately preceding key. Both are averaged over sequences and reported per head, in [0, 1].

**Zero-ablation** sets a head's `hook_z` output to zero for every position, removing its contribution to the residual stream. This is off-distribution: the model never saw activations like this during training, so ablation effects should be read as evidence about what a head contributes under normal operation, not as a clean causal intervention.

**Control heads** are chosen at random, disjoint from the ablated set, with the same per-layer counts, to guard against the confound that ablating any heads (not just induction heads) might hurt performance. Their scores are recorded alongside the ablated heads' scores rather than screened by any exclusion rule.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root / "src"))
from copy_lab.heads import run


## Experimental design

Using the same `BOS A A` sequences as the copying baseline, score every head's induction and previous-token attention, rank heads by induction score, and zero-ablate the top-`k` induction heads. Compare the resulting paired copying benefit (control minus repeated-context loss on the second block) against the unablated baseline and against ablating an equal number of random control heads per layer.

Configuration: 32 tokens per block, 16 sequence pairs, seed 42, top-2 induction heads, CPU execution.

In [ ]:
summary = run(length=32, samples=16, seed=42, top_k=2)

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(root / "results/heads/heads.png")))
print("Top 4 ranked heads:")
for h in summary["ranked_heads"][:4]:
    print(h)
for condition in ("baseline", "induction_ablated", "control_ablated"):
    benefit = summary["ablations"][condition]["paired_copying_benefit_nats"]
    print(condition, "benefit:", benefit, "nats")


## Interpretation and limitations

A large drop in paired copying benefit under `induction_ablated` relative to both `baseline` and `control_ablated` supports the claim that the top-ranked heads causally contribute to the copying behavior, beyond the generic effect of removing any heads. A small or absent gap does not rule out a distributed circuit spread across more heads than `top_k` captures.

Zero-ablation pushes the model off-distribution, so the size of the effect should not be read as a precise causal estimate. Control heads are drawn at random per run (seeded for reproducibility); a different seed would select different controls and could change the control benefit by chance.

Random token inputs still differ from natural language, and this experiment inherits the baseline's other limitations. Further work should test sensitivity to sequence length, random seed, `top_k`, and ablation method (e.g. mean ablation instead of zero).

Rerunning overwrites the files in `results/heads/`; preserve each run separately when comparing configurations.

## References

- [TransformerLens main demo](https://transformerlensorg.github.io/TransformerLens/generated/demos/Main_Demo.html): pretrained models and repeated-sequence induction analysis.
- [ARENA transformer interpretability](https://learn.arena.education/chapter1_transformer_interp/02_intro_mech_interp/): induction circuits and experimental methods.